# 03 — Induction-head discovery on a small transformer

**Programme 03 — Circuits and the Biology of LLMs.** [Programme file.](../programmes/03-circuits-and-biology.md)

By the end of this notebook you will have evaluated each attention head of GPT-2-small on a synthetic *random-prefix-repeats* task, plotted the per-head induction signature (Olsson et al. 2022, [arXiv:2209.11895](https://arxiv.org/abs/2209.11895)), seen the two-three heads in the second layer light up strongly, and confirmed causally by ablating them and watching the task loss jump. The claim being demonstrated is that small transformers contain identifiable, causally-validated *induction heads* — the canonical circuit motif of mechanistic interpretability.

What this notebook is *not*: a finding. Olsson et al. did this at scale across many model families and showed the universality. Here we replicate the qualitative signature on a single small model in a few minutes.

In [ ]:
# Install pinned versions (Colab).
# %pip install -q torch==2.4.1 transformer-lens==2.6.0 numpy==1.26.4 matplotlib==3.9.2

In [ ]:
import os, math
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
os.makedirs('figures/03', exist_ok=True)

model = HookedTransformer.from_pretrained('gpt2', device=DEVICE)
model.eval()
print('model:', model.cfg.model_name, 'layers:', model.cfg.n_layers, 'heads/layer:', model.cfg.n_heads)

## A synthetic repeated-prefix task

We sample random sequences of length $L$, then *concatenate them to themselves* to form length $2L$ sequences. The second half is fully predictable from the first half: token at position $L + i$ should match token at position $i$. A head that implements the induction pattern *"attend from position $j$ to position $i$ where the previous occurrence of `tok[j]` was at $i-1$, copy `tok[i]`"* will excel on this.

In [ ]:
L = 32
BATCH = 8
VOCAB = model.cfg.d_vocab

def sample_repeated(batch=BATCH, L=L):
    # Avoid special tokens by sampling from a safe band.
    rng = np.random.default_rng(SEED)
    half = rng.integers(low=1000, high=VOCAB - 1000, size=(batch, L))
    full = np.concatenate([half, half], axis=1)
    return torch.tensor(full, dtype=torch.long, device=DEVICE)

TOKS = sample_repeated()
print('input shape:', tuple(TOKS.shape))

## Per-head induction score

For each attention head and each position $p$ in the *second half* of the sequence, we measure the attention paid to the *induction-target position*: position $p - L + 1$, which is the token *after* the previous occurrence of `tok[p]`. An induction head has attention concentrated there; a non-induction head does not.

**Prediction.** A small number of heads in GPT-2-small (typically in layers 5–10) should score notably higher than the rest.

In [ ]:
with torch.no_grad():
    _, cache = model.run_with_cache(TOKS, return_type='loss', remove_batch_dim=False)

n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads
scores = np.zeros((n_layers, n_heads))

for layer in range(n_layers):
    attn = cache['pattern', layer]  # (batch, head, query, key)
    # second-half query positions
    qs = list(range(L, 2 * L))
    # corresponding induction-target key positions: q - L + 1
    ks = [q - L + 1 for q in qs]
    ks_t = torch.tensor(ks, device=DEVICE)
    qs_t = torch.tensor(qs, device=DEVICE)
    induction = attn[:, :, qs_t, ks_t]  # (batch, head, len(qs))
    scores[layer] = induction.mean(dim=(0, 2)).cpu().numpy()

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(scores, aspect='auto', origin='lower')
ax.set_xlabel('head index'); ax.set_ylabel('layer')
ax.set_title('per-head induction score (random-prefix-repeats, GPT-2)')
fig.colorbar(im)
fig.tight_layout()
fig.savefig('figures/03/induction_scores.png', dpi=120)
plt.show()

topk = np.dstack(np.unravel_index(np.argsort(-scores, axis=None), scores.shape))[0][:5]
print('top-5 induction heads (layer, head):')
for (l, h) in topk:
    print(f'  L{l}H{h}  score={scores[l, h]:.3f}')

## Causal validation by head-ablation

**Prediction.** Mean-ablating the top induction heads (replacing their output with the batch-mean) should *increase* the loss on the second half of the sequence noticeably more than ablating a similar number of randomly chosen heads.

In [ ]:
def loss_second_half(toks):
    logits = model(toks, return_type='logits')
    # cross-entropy of predicting tok[t+1] given tok[0..t], averaged over second-half t.
    log_probs = logits.log_softmax(dim=-1)
    losses = []
    for t in range(L, 2 * L - 1):
        lp = log_probs[:, t, :].gather(-1, toks[:, t + 1:t + 2]).squeeze(-1)
        losses.append(-lp.mean().item())
    return float(np.mean(losses))

import contextlib

@contextlib.contextmanager
def ablate_heads(head_list):
    """Mean-ablate specified attention heads (layer, head)."""
    hooks = []
    def make_hook(h_idx):
        def hook(z, hook):
            mean = z[:, :, h_idx, :].mean(dim=(0, 1), keepdim=True)
            z[:, :, h_idx, :] = mean
            return z
        return hook
    for (layer, head) in head_list:
        hooks.append((f'blocks.{layer}.attn.hook_z', make_hook(head)))
    try:
        for name, hook in hooks:
            model.add_hook(name, hook)
        yield
    finally:
        model.reset_hooks()

baseline = loss_second_half(TOKS)
print(f'baseline loss (second half) = {baseline:.3f}')

with ablate_heads([tuple(x) for x in topk[:3]]):
    ablated = loss_second_half(TOKS)
print(f'after ablating top-3 induction heads = {ablated:.3f}  (delta = {ablated - baseline:+.3f})')

rng = np.random.default_rng(SEED + 1)
random_heads = [(int(rng.integers(0, n_layers)), int(rng.integers(0, n_heads))) for _ in range(3)]
with ablate_heads(random_heads):
    random_ablated = loss_second_half(TOKS)
print(f'after ablating 3 random heads        = {random_ablated:.3f}  (delta = {random_ablated - baseline:+.3f})')
print('random heads chosen:', random_heads)

## What this shows. What this does not show. What would refute it.

**What this shows.** Two-to-three attention heads in GPT-2-small consistently score much higher than the rest on the induction signature on a synthetic repeated-prefix task. Mean-ablating them increases loss on the second half of the sequence by substantially more than ablating an equal number of random heads — *causal* evidence that those heads do work on the task.

**What this does not show.** A synthetic repeated-prefix task is the *easiest* setting in which induction heads show up. On natural text, ICL is supported by induction heads *plus* additional circuitry (task-vector composition, name-mover heads on specific tasks, etc.). The Olsson et al. 2022 contribution was establishing the universality of induction heads across model scales and connecting their formation to the *onset* of in-context learning during training — neither of which is shown here.

**What would refute the toy claim.** No head in any layer scores measurably above baseline on the induction signature; or, the top-scoring heads under the signature do *not* matter causally (ablating them does not move loss more than random ablation does).

**Where to go next.** Read Olsson et al. (2022) end-to-end. Their evidence base — multiple model scales, induction-head formation as a phase change during training, connection to ICL onset — is what makes this finding load-bearing for both Programme 03 and Programme 04.